In [16]:
import pasta
import numpy as np 
import nibabel as nib 
from pathlib import Path 
from scipy.spatial.distance import squareform, pdist

### first test memory efficiency with 32k mesh

In [ ]:
# example generated data at 32k spherical mesh
data_path = Path("test_data")
data0 = data_path / "0_0.shape.gii"
data1 = data_path / "0_1.shape.gii"
mesh = data_path / "tpl-fsLR_den-32k_hemi-L_sphere.surf.gii"
x = nib.load(data0).darrays[0].data
y = nib.load(data1).darrays[0].data
vertices = nib.load(mesh).darrays[0].data
D = squareform(pdist(vertices))
triu_indices = np.triu_indices_from(D, k=1)
D_upper = D[triu_indices]

In [ ]:
# 1. try default setting with Euclidean distance on the fly
# 1.1 pasta: 2m20s Mac M1 Pro
# pasta(x, y, coord, *, M=None, M_scale=3, qd=0.7, nugget=True, kernel_scale=4.0, block_size=2000, n_workers=1, xparc=None, yparc=None, max_clusters=10, min_clusters=1, min_cluster_size=500, D=None, dim=None, random_state=None) -> PaSTAFit:
pastafit = pasta.pasta(x, y, vertices)
print(pastafit)

In [ ]:
# 1.2 pasta-ns: 3m17s Mac M1 Pro
pastafit_ns = pasta.pasta(x, y, vertices, xparc="auto", yparc="auto")
print(pastafit_ns)

In [ ]:
# 2. try with provided distance matrix
# 2.1 pasta: 3m Mac M1 Pro
pastafit_withD = pasta.pasta(x, y, vertices, D=D)
print(pastafit_withD)

In [ ]:
# 2.2 pasta-ns: 3m22s Mac M1 pro
pastafit_ns_withD = pasta.pasta(x, y, vertices, xparc="auto", yparc="auto", D=D)
print(pastafit_ns_withD)

In [ ]:
# 3. try with provided upper triangular distance vector
# 3.1 pasta: 2m36s Mac M1 Pro
pastafit_withDupper = pasta.pasta(x, y, vertices, D=D_upper)
# print(pastafit_withDupper)

In [ ]:
# 3.2 pasta-ns: 3m31s Mac M1 pro
pastafit_ns_withDupper = pasta.pasta(x, y, vertices, xparc="auto", yparc="auto", D=D_upper)
# print(pastafit_ns_withDupper)

### Memory efficiency test completed. Now let's test nonstationarity and missing data handling with 10k

In [59]:
# example generated data (nonstationary) at 10k spherical mesh
data_path = Path("test_data")
data0 = data_path / "1_0_ns10k.shape.gii"
data1 = data_path / "1_1_ns10k.shape.gii"
mesh = data_path / "tpl-fsaverage_den-10k_hemi-L_sphere.surf.gii"
x = nib.load(data0).darrays[0].data
y = nib.load(data1).darrays[0].data
vertices = nib.load(mesh).darrays[0].data
D = squareform(pdist(vertices))
triu_indices = np.triu_indices_from(D, k=1)
D_upper = D[triu_indices]

##### compare statioanry and nonstationary results, including rng seed testing

In [ ]:
# 1. stationary: 13s at 10k
pastafit = pasta.pasta(x, y, vertices)

# print(pastafit)

In [ ]:
# 2. nonstationary: 19s at 10k
pastafit_ns = pasta.pasta(x, y, vertices, xparc="auto", yparc="auto")

In [ ]:
# 3. nonstationary with seed reproducibility
pastafit_ns_seed1 = pasta.pasta(x, y, vertices, xparc="auto", yparc="auto", random_state=123)
pastafit_ns_seed2 = pasta.pasta(x, y, vertices, xparc="auto", yparc="auto", random_state=123)

In [67]:
print(pastafit.nef, pastafit_ns.nef, pastafit_ns_seed1.nef, pastafit_ns_seed2.nef)

261.1144483504812 124.02594161694766 160.86022106503714 160.86022106503714


##### test for handling of missing values

In [71]:
nan_idx = np.random.choice(len(x), size=1000, replace=False)

x_select = x.copy()
x_select[nan_idx] = np.nan
y_select = y.copy()
y_select[nan_idx] = np.nan
D = squareform(pdist(vertices))
triu_indices = np.triu_indices_from(D, k=1)
D_upper = D[triu_indices]

In [75]:
pastafit = pasta.pasta(x_select, y_select, vertices)

In [77]:
pastafit_ns = pasta.pasta(x_select, y_select, vertices, xparc="auto", yparc="auto")

In [84]:
pastafit_ns_seed1 = pasta.pasta(x_select, y_select, vertices, xparc="auto", yparc="auto", random_state=123)
pastafit_ns_seed2 = pasta.pasta(x_select, y_select, vertices, xparc="auto", yparc="auto", random_state=123)

In [85]:
print(pastafit.nef, pastafit_ns.nef, pastafit_ns_seed1.nef, pastafit_ns_seed2.nef)

261.0980554078078 196.10477498131155 137.7456311777512 137.7456311777512
